# Step 10: Train All µP Model Sizes
**Prerequisite**: Run `colab_09_mup_lr_sweep.py` first to get the best LR.
Expects data at `MyDrive/ML_project/data/splits/` and sweep results at
`MyDrive/ML_project/checkpoints/mup_lr_sweep/sweep_results.json`

In [1]:
!pip install -q mup
import os, sys, json, time, math, shutil
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from mup import MuSharedReadout, set_base_shapes
from mup.optim import MuAdamW

from google.colab import drive
drive.mount('/content/drive')

BASE = Path("/content/ML_project"); BASE.mkdir(exist_ok=True)
DATA = BASE/"data"/"splits"; DATA.mkdir(parents=True, exist_ok=True)
CKPT = BASE/"checkpoints"/"mup_scaling"; CKPT.mkdir(parents=True, exist_ok=True)

DRIVE_DATA = Path("/content/drive/MyDrive/ML_project/data/splits")
for f in ["train.npy", "val.npy"]:
    src, dst = DRIVE_DATA/f, DATA/f
    if src.exists() and not dst.exists(): shutil.copy2(src, dst); print(f"Copied {f}")

# print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")

  Preparing metadata (setup.py) ... done
Mounted at /content/drive
Copied train.npy
Copied val.npy


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [2]:
@dataclass
class GPTConfig:
    block_size: int = 2048; vocab_size: int = 4096; n_layer: int = 6
    n_head: int = 6; n_embd: int = 384; d_ff: int = None
    dropout: float = 0.0; bias: bool = False

MODEL_CONFIGS = {
    "tiny": GPTConfig(n_layer=4, n_head=4, n_embd=128, d_ff=512),
    "small": GPTConfig(n_layer=6, n_head=6, n_embd=192, d_ff=768),
    "medium": GPTConfig(n_layer=6, n_head=6, n_embd=384, d_ff=1536),
    "large": GPTConfig(n_layer=10, n_head=8, n_embd=512, d_ff=2048),
    "xl": GPTConfig(n_layer=12, n_head=12, n_embd=768, d_ff=3072),
}

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class MuCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.head_dim = config.n_embd // config.n_head; self.dropout = config.dropout
        self.q_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B,T,self.n_head,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_head,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_head,self.head_dim).transpose(1,2)
        y = F.scaled_dot_product_attention(q,k,v,attn_mask=None,
            dropout_p=self.dropout if self.training else 0, is_causal=True, scale=8.0/self.head_dim)
        return self.resid_dropout(self.c_proj(y.transpose(1,2).contiguous().view(B,T,C)))

class MuMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_ff = config.d_ff or 4*config.n_embd
        self.c_fc = nn.Linear(config.n_embd, d_ff, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(d_ff, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x): return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class MuBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, config.bias)
        self.attn = MuCausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MuMLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x)); return x + self.mlp(self.ln_2(x))

class MuGPT(nn.Module):
    def __init__(self, config):
        super().__init__(); self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([MuBlock(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias)))
        self.lm_head = MuSharedReadout(self.transformer.wte.weight, bias=False)
    def _init_weights_mup(self):
        for _, m in self.named_modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 1.0/math.sqrt(m.weight.shape[1]))
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0, 0.02)
        for b in self.transformer.h:
            nn.init.zeros_(b.attn.q_proj.weight)
            for p in [b.attn.c_proj, b.mlp.c_proj]:
                nn.init.normal_(p.weight, 0, 1.0/math.sqrt(p.weight.shape[1])/math.sqrt(2*self.config.n_layer))
    def get_num_params(self):
        return sum(p.numel() for p in self.parameters()) - self.transformer.wpe.weight.numel()
    def forward(self, idx, targets=None):
        b,t = idx.size(); pos = torch.arange(t, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            return logits, F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1), ignore_index=-1)
        return self.lm_head(x[:, [-1], :]), None

def create_mup_model(name):
    cfg = MODEL_CONFIGS[name]; nh = cfg.n_head
    bc = GPTConfig(n_layer=cfg.n_layer, n_head=nh, n_embd=nh, d_ff=nh*4, block_size=cfg.block_size, vocab_size=cfg.vocab_size, bias=cfg.bias)
    dc = GPTConfig(n_layer=cfg.n_layer, n_head=nh, n_embd=nh*2, d_ff=nh*8, block_size=cfg.block_size, vocab_size=cfg.vocab_size, bias=cfg.bias)
    bm, dm, m = MuGPT(bc), MuGPT(dc), MuGPT(cfg)
    set_base_shapes(m, bm, delta=dm); m._init_weights_mup(); del bm, dm; return m

In [3]:
def get_batch(data, bs, bsz, dev):
    ix = torch.randint(len(data)-bs, (bsz,))
    x = torch.stack([torch.from_numpy(data[i:i+bs].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+bs].astype(np.int64)) for i in ix])
    return x.to(dev), y.to(dev)

@torch.no_grad()
def evaluate(model, data, bs, bsz, n, dev, ctx):
    model.eval(); L = []
    for _ in range(n):
        x, y = get_batch(data, bs, bsz, dev)
        with ctx: _, l = model(x, y); L.append(l.item())
    model.train(); return float(np.mean(L))

def get_lr(step, warmup, mx, hi, lo):
    if step < warmup: return hi*(step+1)/warmup
    if step >= mx: return lo
    return lo + 0.5*(1+math.cos(math.pi*(step-warmup)/(mx-warmup)))*(hi-lo)

def train_mup(name, lr, mbs, ga, tp, vp, warmup=200, ckpt_dir=None):
    dev="cuda"; ctx=torch.amp.autocast(device_type="cuda",dtype=torch.bfloat16)
    td, vd = np.load(tp, mmap_mode='r'), np.load(vp, mmap_mode='r')
    bs=2048; tps=mbs*ga*bs; total=len(td)//tps
    model = create_mup_model(name).to(dev); np_ = model.get_num_params()
    print(f"  {name}: {np_:,} ({np_/1e6:.1f}M), LR={lr}, steps={total}")
    pd={pn:p for pn,p in model.named_parameters() if p.requires_grad}
    dec=[p for n,p in pd.items() if p.dim()>=2]; nod=[p for n,p in pd.items() if p.dim()<2]
    opt = MuAdamW([{'params':dec,'weight_decay':0.1},{'params':nod,'weight_decay':0.0}], lr=lr, betas=(0.9,0.95))
    scaler=torch.amp.GradScaler(enabled=False); mlr=lr*0.1; ilrs=[pg['lr'] for pg in opt.param_groups]
    model.train(); t0=time.time(); ttok=0; bvl=float('inf'); tls,vls=[],[]
    for step in range(total):
        blr=get_lr(step,warmup,total,lr,mlr); r=blr/lr
        for pg,il in zip(opt.param_groups,ilrs): pg['lr']=il*r
        opt.zero_grad(set_to_none=True); al=0
        for _ in range(ga):
            x,y=get_batch(td,bs,mbs,dev)
            with ctx: _,l=model(x,y); l=l/ga
            scaler.scale(l).backward(); al+=l.item(); ttok+=x.numel()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt); scaler.update()
        if (step+1)%10==0: tls.append((step,al))
        if (step+1)%50==0 or step==total-1:
            vl=evaluate(model,vd,bs,mbs,20,dev,ctx); vls.append((step,vl))
            if vl<bvl: bvl=vl
            print(f"    step {step+1}/{total} lr={blr:.6f} val={vl:.4f} best={bvl:.4f}")
    wall=time.time()-t0; ftl=tls[-1][1] if tls else 0; fvl=vls[-1][1] if vls else 0
    print(f"  Done {wall:.0f}s best_val={bvl:.4f} tok/s={ttok/wall:,.0f}")
    res={"model_name":name,"n_params":np_,"learning_rate":lr,"best_val_loss":bvl,
         "final_val_loss":fvl,"final_train_loss":ftl,"wall_time_seconds":wall,
         "tokens_per_second":ttok/wall,"parameterization":"mup","train_losses":tls,"val_losses":vls,
         "config":{"n_layer":MODEL_CONFIGS[name].n_layer,"n_head":MODEL_CONFIGS[name].n_head,
                   "n_embd":MODEL_CONFIGS[name].n_embd,"d_ff":MODEL_CONFIGS[name].d_ff},
         "peak_gpu_memory_mb":torch.cuda.max_memory_allocated()/1e6}
    if ckpt_dir:
        cd=Path(ckpt_dir)/name; cd.mkdir(parents=True,exist_ok=True)
        with open(cd/"metrics.json",'w') as f: json.dump(res,f,indent=2,default=str)
        torch.save({'model_state_dict':model.state_dict(),'metrics':res},cd/"model.pt")
    del model,opt; torch.cuda.empty_cache()
    return res

In [4]:
sweep_path = Path("/content/drive/MyDrive/ML_project/checkpoints/mup_lr_sweep/sweep_results.json")
with open(sweep_path) as f: best_lr = json.load(f)["best_lr"]
print(f"Using best muP LR: {best_lr}")

TP, VP = str(DATA/"train.npy"), str(DATA/"val.npy")

# A100 batch configs (40GB VRAM)
A100_CONFIGS = {
    "tiny":   {"mbs": 64,  "ga": 4},
    "small":  {"mbs": 32,  "ga": 8},
    "medium": {"mbs": 16,  "ga": 16},
    "large":  {"mbs": 8,   "ga": 32},
    "xl":     {"mbs": 4,   "ga": 64},
}

all_results = []
for name, bc in A100_CONFIGS.items():
    print(f"\n{'#'*50}\n  Training: {name.upper()}\n{'#'*50}")
    r = train_mup(name, best_lr, bc["mbs"], bc["ga"], TP, VP, warmup=200, ckpt_dir=str(CKPT))
    all_results.append(r)

Using best muP LR: 0.1

##################################################
  Training: TINY
##################################################
  tiny: 1,311,872 (1.3M), LR=0.1, steps=281
    step 50/281 lr=0.025000 val=4.1558 best=4.1558
    step 100/281 lr=0.050000 val=2.2239 best=2.2239
    step 150/281 lr=0.075000 val=2.1516 best=2.1516
    step 200/281 lr=0.100000 val=1.9615 best=1.9615
    step 250/281 lr=0.040432 val=1.7767 best=1.7767
    step 281/281 lr=0.010034 val=1.6485 best=1.6485
  Done 61s best_val=1.6485 tok/s=2,401,599

##################################################
  Training: SMALL
##################################################
  small: 3,443,136 (3.4M), LR=0.1, steps=281
    step 50/281 lr=0.025000 val=3.9047 best=3.9047
    step 100/281 lr=0.050000 val=2.2086 best=2.2086
    step 150/281 lr=0.075000 val=2.1553 best=2.1553
    step 200/281 lr=0.100000 val=2.0373 best=2.0373
    step 250/281 lr=0.040432 val=1.7979 best=1.7979
    step 281/281 lr=0.010034 val=1

In [5]:
print(f"\n{'='*60}\nALL muP MODELS TRAINED\n{'='*60}")
print(f"{'Model':<8} {'Params':>12} {'Val Loss':>10} {'Time':>8} {'Tok/s':>10}")
print("-"*50)
for r in all_results:
    print(f"{r['model_name']:<8} {r['n_params']:>12,} {r['best_val_loss']:>10.4f} {r['wall_time_seconds']:>7.0f}s {r['tokens_per_second']:>10,.0f}")

with open(CKPT/"all_results.json", 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

# Copy to Drive
DRIVE_CKPT = Path("/content/drive/MyDrive/ML_project/checkpoints/mup_scaling")
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT/"all_results.json", DRIVE_CKPT/"all_results.json")
for name in A100_CONFIGS:
    src_dir = CKPT/name; dst_dir = DRIVE_CKPT/name; dst_dir.mkdir(exist_ok=True)
    for f in ["metrics.json"]:  # skip model.pt (too large for Drive)
        if (src_dir/f).exists(): shutil.copy2(src_dir/f, dst_dir/f)
print(f"All results saved to Drive: {DRIVE_CKPT}")


ALL muP MODELS TRAINED
Model          Params   Val Loss     Time      Tok/s
--------------------------------------------------
tiny        1,311,872     1.6485      61s  2,401,599
small       3,443,136     1.6461     115s  1,285,670
medium     12,194,688     1.6187     191s    773,072
large      33,565,184     1.6991     428s    344,335
xl         88,099,584     1.8295     848s    173,827
All results saved to Drive: /content/drive/MyDrive/ML_project/checkpoints/mup_scaling
